In [1]:
import numpy as np
from sklearn import mixture
#import torch
#import triton
#import triton.language as tl

np.random.seed(43)
#torch.manual_seed(42)

#DEVICE = triton.runtime.driver.active.get_active_torch_device()

In [2]:
n_models = 2
n_components = 3
n_gaussians = n_components * n_models
true_means = np.random.rand(n_gaussians, 2) * 10
true_sigmas = np.array([np.eye(2) * (0.1 + 0.9 * np.random.rand()) for _ in range(n_gaussians)])
true_weights = np.hstack([np.random.dirichlet(np.ones(n_components), size=1)[0] for _ in range(n_models)])
model_labels = np.repeat(np.arange(n_models), n_components)

n_samples = 50
sections = []
labels = []
for i in range(len(true_weights)):
    X_i = np.random.multivariate_normal(mean=true_means[i], cov=true_sigmas[i], size=int(n_samples * true_weights[i]))
    sections.append(X_i)
    labels.append(np.full(X_i.shape[0], model_labels[i]))
X = np.vstack(sections)
labels = np.hstack(labels)

offsets = [0]

sectioned_points = []
for label in range(n_models):
    points = X[labels == label]
    sectioned_points.append(points.copy())
    offsets.append(points.shape[0] + offsets[label])

offsets = np.array(offsets)

#X_torch = torch.from_numpy(X).float()
#X_torch_cuda = X_torch.cuda()

In [3]:
from kmeans import KMeans

kmeans_models = []
for section in sectioned_points:
    kmeans_model = KMeans(n_clusters=n_components, max_iter=1000, verbose=True)
    kmeans_model.fit(section)
    kmeans_models.append(kmeans_model)

Iteration: 0: Progress: 1.4040392871326077
Iteration: 1: Progress: 0.0
Iteration: 0: Progress: 2.1240401631560615
Iteration: 5: Progress: 0.0


In [4]:
from warp_kmeans import WarpKMeans

warp_model = WarpKMeans(n_models=n_models, n_clusters=n_components)
warp_model.fit(X, labels, offsets)

Warp 1.15.0 initialized:
   CUDA Toolkit 12.9, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA GeForce RTX 3080" (10 GiB, sm_86, mempool enabled)
   Kernel cache:
     /localhome/dre3/.cache/warp/1.15.0
Module warp_kmeans 31a25d7 load on device 'cuda:0' took 0.90 ms  (cached)


In [5]:
from tiled_kmeans import TiledKMeans

tiled_model = TiledKMeans(n_models=n_models, n_clusters=n_components)
tiled_model.fit(X, labels, offsets)

Init took: 0.0002894401550292969s
Module tiled_kmeans 00ca651 load on device 'cuda:0' took 0.96 ms  (cached)


In [6]:
[m.centroids for m in kmeans_models], warp_model.centroids, tiled_model.centroids

([array([[1.20987488, 2.59347279],
         [4.52795801, 8.55269855],
         [2.68315576, 8.18935339]]),
  array([[7.21119664, 5.08617005],
         [3.72547629, 7.95581177],
         [6.27849715, 5.54101551]])],
 array([[2.7327874, 8.271543 ],
        [1.2098747, 2.593473 ],
        [4.6782575, 8.352986 ],
        [6.7059846, 5.3325443],
        [4.156821 , 8.003729 ],
        [0.706067 , 7.6203904]], dtype=float32),
 array([[1.2098747, 2.593473 ],
        [3.8632388, 7.8926153],
        [2.480867 , 8.722772 ],
        [4.15682  , 8.00373  ],
        [6.7059846, 5.3325443],
        [0.706067 , 7.6203904]], dtype=float32))

In [7]:
true_means

array([[1.15054566, 6.09066539],
       [1.33390964, 2.4058962 ],
       [3.27139056, 8.59137491],
       [6.66090213, 5.41162212],
       [0.29013824, 7.33748296],
       [3.94950018, 8.02047119]])